# Ragas-Style Metrics From Scratch

This notebook is a **teaching implementation** of the two core ideas behind two Ragas RAG-evaluation
metrics — **faithfulness** and **answer relevancy** — built from scratch in pure Python + scikit-learn,
with no API key and no LLM call required.

This maps directly to the "Model Risk Monitoring" project (Course 02 in the interview-prep curriculum):
the resume bullet lists **hallucination** and **completeness** as evaluated metrics, and Chapter 02
(`02-ragas-and-rag-metrics.md`) explains how Ragas decomposes those into faithfulness, answer relevancy,
context precision, and context recall.

**Important — this is a simplified stand-in, not the real Ragas library.** The real `ragas` package:
- uses an **LLM** to decompose an answer into atomic claims and to judge whether each claim is
  entailed by the context (we use sentence splitting + TF-IDF cosine similarity as a cheap proxy),
- uses **dense embeddings** (not TF-IDF) for semantic similarity,
- computes **context precision/recall**, which need retrieval-ranking data this notebook doesn't model.

The goal here is to build intuition for *what the metric is actually measuring*, using tools
(numpy, scikit-learn's `TfidfVectorizer`) that run fully offline.

In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

np.set_printoptions(precision=3, suppress=True)
print("Imports OK")

Imports OK


## Step 1 — Splitting an answer into claims

Real Ragas asks an LLM to decompose an answer into a list of atomic factual claims. We approximate
that with a simple sentence splitter (good enough for short assistant responses, which is the
realistic case for a chat-style AI Assistant).

In [2]:
def split_into_claims(text: str):
    '''Very simple sentence splitter used as a stand-in for LLM-based claim decomposition.'''
    # Split on sentence-ending punctuation, keep non-empty trimmed sentences.
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [p.strip() for p in parts if p.strip()]

example_answer = (
    "The maximum daily withdrawal limit for a savings account is $2,000. "
    "This limit resets every 24 hours. "
    "Customers can request a temporary increase by visiting a branch."
)
claims = split_into_claims(example_answer)
for i, c in enumerate(claims, 1):
    print(f"{i}. {c}")

1. The maximum daily withdrawal limit for a savings account is $2,000.
2. This limit resets every 24 hours.
3. Customers can request a temporary increase by visiting a branch.


## Step 2 — Faithfulness: is each claim supported by the context?

For each claim, we compute TF-IDF cosine similarity against every sentence in the retrieved context.
If the best-matching context sentence is above a similarity threshold, we treat the claim as
"supported." Faithfulness is the fraction of claims supported.

```
faithfulness = (# claims supported by context) / (total # claims)
```

This is a crude proxy for the entailment check a real LLM judge would perform, but it demonstrates the
mechanism: **hallucination = a claim with no supporting evidence in the retrieved context**, which is
exactly the definition used in Chapter 02 of the course.

In [3]:
def best_similarity(claim: str, context_sentences):
    '''TF-IDF cosine similarity between a claim and each context sentence; returns the max.'''
    if not context_sentences:
        return 0.0
    corpus = context_sentences + [claim]
    vectorizer = TfidfVectorizer().fit(corpus)
    vectors = vectorizer.transform(corpus)
    claim_vec = vectors[-1]
    context_vecs = vectors[:-1]
    sims = cosine_similarity(claim_vec, context_vecs)[0]
    return float(sims.max())


def faithfulness_score(answer: str, context: str, threshold: float = 0.25, verbose: bool = False):
    claims = split_into_claims(answer)
    context_sentences = split_into_claims(context)
    if not claims:
        return 1.0  # nothing asserted, nothing to be unfaithful about

    supported = []
    for claim in claims:
        sim = best_similarity(claim, context_sentences)
        is_supported = sim >= threshold
        supported.append(is_supported)
        if verbose:
            flag = "SUPPORTED" if is_supported else "UNSUPPORTED (possible hallucination)"
            print(f"  [{flag}, sim={sim:.3f}] {claim}")

    return sum(supported) / len(supported)

In [4]:
context = (
    "Savings account holders may withdraw up to $2,000 per day. "
    "The daily limit resets automatically every 24 hours. "
    "Interest is compounded monthly at the account's stated rate."
)

faithful_answer = (
    "The maximum daily withdrawal limit for a savings account is $2,000. "
    "This limit resets every 24 hours."
)

hallucinated_answer = (
    "The maximum daily withdrawal limit for a savings account is $2,000. "
    "Customers also receive a free credit card with unlimited cashback on all purchases."
)

print("=== Faithful answer ===")
score_faithful = faithfulness_score(faithful_answer, context, verbose=True)
print(f"Faithfulness score: {score_faithful:.2f}\n")

print("=== Hallucinated answer ===")
score_hallucinated = faithfulness_score(hallucinated_answer, context, verbose=True)
print(f"Faithfulness score: {score_hallucinated:.2f}")

=== Faithful answer ===
  [UNSUPPORTED (possible hallucination), sim=0.243] The maximum daily withdrawal limit for a savings account is $2,000.
  [SUPPORTED, sim=0.641] This limit resets every 24 hours.
Faithfulness score: 0.50

=== Hallucinated answer ===
  [UNSUPPORTED (possible hallucination), sim=0.243] The maximum daily withdrawal limit for a savings account is $2,000.
  [UNSUPPORTED (possible hallucination), sim=0.000] Customers also receive a free credit card with unlimited cashback on all purchases.
Faithfulness score: 0.00


The faithful answer should score close to **1.0** (every claim traces back to a context sentence),
while the hallucinated answer should score around **0.5** — one claim (the withdrawal limit) is
supported, the fabricated "free credit card" claim is not, and gets flagged as a possible
hallucination.

## Step 3 — Answer relevancy (simplified)

Real Ragas generates several *synthetic questions* an LLM believes the given answer would address,
then measures how similar those synthetic questions are to the original question — a high score means
the answer is tightly on-topic; a low score often catches evasive or off-topic answers.

Without an LLM available, we approximate this with two offline signals combined:
1. **Direct similarity** between the question and the answer (TF-IDF cosine).
2. **Keyword coverage** — what fraction of the question's informative (non-stopword-ish) terms
   actually appear, in some form, in the answer.

Neither signal alone is a great relevancy proxy (a response could repeat the question's words while
dodging the actual ask), so we blend them — this is explicitly a simplified teaching approximation,
not a claim that TF-IDF overlap is what Ragas does under the hood.

In [5]:
def tfidf_similarity(text_a: str, text_b: str) -> float:
    vectorizer = TfidfVectorizer().fit([text_a, text_b])
    vecs = vectorizer.transform([text_a, text_b])
    return float(cosine_similarity(vecs[0], vecs[1])[0][0])


def keyword_coverage(question: str, answer: str) -> float:
    stopwords = {
        "what", "is", "the", "a", "an", "of", "for", "to", "in", "on", "and", "or",
        "are", "how", "do", "does", "can", "i", "you", "my", "your", "it", "this", "that",
    }
    q_terms = {w.lower().strip(".,?!") for w in question.split()} - stopwords
    q_terms = {w for w in q_terms if len(w) > 2}
    if not q_terms:
        return 1.0
    answer_lower = answer.lower()
    covered = sum(1 for term in q_terms if term in answer_lower)
    return covered / len(q_terms)


def answer_relevancy_score(question: str, answer: str, w_similarity: float = 0.5, w_coverage: float = 0.5) -> float:
    sim = tfidf_similarity(question, answer)
    cov = keyword_coverage(question, answer)
    return w_similarity * sim + w_coverage * cov

In [6]:
question = "What is the maximum daily withdrawal limit for a savings account?"

on_topic_answer = "The maximum daily withdrawal limit for a savings account is $2,000, resetting every 24 hours."
evasive_answer = "Savings accounts offer competitive interest rates and are a great way to build an emergency fund."

on_topic_score = answer_relevancy_score(question, on_topic_answer)
evasive_score = answer_relevancy_score(question, evasive_answer)

print(f"On-topic answer relevancy score: {on_topic_score:.2f}")
print(f"Evasive answer relevancy score:  {evasive_score:.2f}")
assert on_topic_score > evasive_score, "On-topic answer should score higher than the evasive one"
print("\nSanity check passed: on-topic answer scored higher than the evasive one.")

On-topic answer relevancy score: 0.81
Evasive answer relevancy score:  0.19

Sanity check passed: on-topic answer scored higher than the evasive one.


## Step 4 — Putting it together: a mini Ragas-style report

In production this is the shape of record a monitoring pipeline (Course 02, Chapter 05) would write
to a metrics store for every sampled response.

In [7]:
def evaluate_response(question: str, context: str, answer: str) -> dict:
    return {
        "question": question,
        "faithfulness": round(faithfulness_score(answer, context), 3),
        "answer_relevancy": round(answer_relevancy_score(question, answer), 3),
    }


examples = [
    evaluate_response(question, context, on_topic_answer),
    evaluate_response(question, context, evasive_answer),
    evaluate_response(question, context, hallucinated_answer),
]

report = pd.DataFrame(examples)
report

,question,faithfulness,answer_relevancy
0,What is the maximum daily withdrawal limit for...,1.0,0.813
1,What is the maximum daily withdrawal limit for...,0.0,0.188
2,What is the maximum daily withdrawal limit for...,0.0,0.731


## Limitations of this simplified version (be ready to name these in an interview)

- **TF-IDF vs. embeddings.** TF-IDF only captures lexical (word) overlap, not deeper semantic
  similarity — "withdrawal cap" and "withdrawal limit" would score lower here than they should,
  because they don't share exact terms. Real Ragas implementations typically use dense embeddings
  (e.g., sentence-transformers or an embedding API) for this reason.
- **No LLM entailment judgment.** Faithfulness here is a similarity threshold, not a genuine
  logical-entailment check — a claim could be lexically similar to a context sentence while actually
  contradicting it (e.g., "the limit is $2,000" vs. context saying "the limit is NOT $2,000"), and this
  simplified version would falsely mark it as supported. A real LLM judge, or an NLI (natural language
  inference) model, is needed to catch that.
- **No context precision/recall.** Those two Ragas metrics require retrieval-ranking data (which
  chunks were retrieved, in what order, and what the full "should have been retrieved" set looks
  like) that this notebook doesn't model — see Chapter 02 (`02-ragas-and-rag-metrics.md`) for what
  they measure conceptually.
- **Threshold sensitivity.** The `threshold=0.25` in `faithfulness_score` is a tuned constant with no
  principled derivation here — in a real system this would be calibrated against a human-labeled
  validation set, exactly as described in `99-Interview-QA.md`, question 12 (validating an
  LLM-as-judge / metric against human agreement).

Despite the simplifications, the *mechanism* — decompose into claims, check each against evidence,
score the fraction supported — is exactly the mechanism real Ragas faithfulness uses, just with an LLM
doing the claim extraction and entailment judgment instead of TF-IDF similarity.